In [0]:
# ============================================================
# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
# MODULE 6 — ACTIONABLE RECOMMENDATIONS ENGINE
# Author: Byron Kamau
# ============================================================
# DEPENDS ON: All previous modules (7, 1, 2, 3, 4)
#
# OUTPUTS:
#   gold_recommendations       — master recommendations table
#   gold_weekly_briefing       — auto-generated trading review
# ============================================================

# Module 6 — Actionable Recommendations Engine
**Layer:** Gold → Gold (cross-module synthesis)

**What this notebook does:**
1. Queries ALL Gold tables and applies business rules
2. Auto-generates flagged commercial issues with evidence
3. Produces a structured recommendations table
4. Generates a weekly trading briefing in markdown format

> This module directly addresses the JD requirement:
> *"You will interrogate performance and recommend action —
> not simply report results"*

## Cell 1 — Setup

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
from datetime import datetime

DATABASE = "uk_retail_commercial"
spark.sql(f"USE {DATABASE}")

print("✅ Database set:", DATABASE)
print("📋 Loading all Gold tables...")

# Load all gold tables
gold_retailer_pl    = spark.table(f"{DATABASE}.gold_retailer_pl")
gold_sku            = spark.table(f"{DATABASE}.gold_sku_profitability")
gold_regional       = spark.table(f"{DATABASE}.gold_regional_performance")
gold_waterfall      = spark.table(f"{DATABASE}.gold_margin_waterfall")
gold_promos         = spark.table(f"{DATABASE}.gold_promo_events")
gold_trade_eff      = spark.table(f"{DATABASE}.gold_trade_spend_efficiency")
gold_promo_rank     = spark.table(f"{DATABASE}.gold_promo_roi_ranking")
gold_forecast       = spark.table(f"{DATABASE}.gold_forecast_scenarios")
gold_accuracy       = spark.table(f"{DATABASE}.gold_forecast_accuracy")
gold_sell_through   = spark.table(f"{DATABASE}.gold_sell_through")
gold_seasonal       = spark.table(f"{DATABASE}.gold_seasonal_risk")
gold_elasticity     = spark.table(f"{DATABASE}.gold_price_elasticity")
gold_competitor     = spark.table(f"{DATABASE}.gold_competitor_pricing")
gold_whitespace     = spark.table(f"{DATABASE}.gold_whitespace")
gold_pricing_recs   = spark.table(f"{DATABASE}.gold_pricing_recommendations")

print("✅ All Gold tables loaded")

✅ Database set: uk_retail_commercial
📋 Loading all Gold tables...
✅ All Gold tables loaded


## Cell 2 — Rule Engine: Margin Flags
Flag retailers and categories where margin is below threshold

In [0]:
rec_list = []
rec_counter = [0]

def add_rec(rec_type, area, issue, evidence, action, impact, priority):
    rec_counter[0] += 1
    rec_list.append({
        "rec_id":             f"REC-{rec_counter[0]:03d}",
        "rec_type":           rec_type,
        "area":               area,
        "issue":              issue,
        "evidence":           evidence,
        "recommended_action": action,
        "commercial_impact":  impact,
        "priority":           priority,
        "status":             "OPEN"
    })

# ── RULE 1: Retailer margin below 30% ──────────────────────
print("🔍 Rule 1: Retailer margin below threshold...")
low_margin_retailers = gold_retailer_pl \
    .groupBy("retailer_name", "channel") \
    .agg(F.avg("avg_contribution_margin_pct").alias("overall_margin_pct")) \
    .filter(F.col("overall_margin_pct") < 30) \
    .collect()

for row in low_margin_retailers:
    add_rec(
        rec_type  = "MARGIN",
        area      = row["retailer_name"],
        issue     = f"Contribution margin below 30% threshold at {row['retailer_name']}",
        evidence  = f"Average contribution margin: {row['overall_margin_pct']:.1f}% (threshold: 30%)",
        action    = f"Review rebate structure and trade funding terms with {row['retailer_name']} — negotiate improved settlement terms or reduce scan allowances",
        impact    = "Every 1% margin improvement on this retailer recovers significant P&L value",
        priority  = "HIGH"
    )
    print(f"  ⚠️  {row['retailer_name']}: {row['overall_margin_pct']:.1f}% margin")

# ── RULE 2: SKUs with AT RISK flag ────────────────────────
print("\n🔍 Rule 2: AT RISK SKUs...")
at_risk_skus = gold_sku \
    .filter(F.col("sku_flag") == "AT RISK") \
    .orderBy("avg_contribution_margin_pct") \
    .limit(5) \
    .collect()

for row in at_risk_skus:
    add_rec(
        rec_type  = "SKU PORTFOLIO",
        area      = f"{row['description']} ({row['category']})",
        issue     = f"SKU '{row['description']}' is AT RISK with {row['avg_contribution_margin_pct']:.1f}% margin",
        evidence  = f"Revenue rank: {row['revenue_rank']} | Units sold: {row['total_units_sold']:,} | Margin rank: {row['margin_rank']}",
        action    = f"Review cost structure or delist '{row['description']}' — consider range rationalisation",
        impact    = "Removing loss-making SKUs improves portfolio average margin",
        priority  = "HIGH"
    )
    print(f"  ⚠️  {row['description']}: {row['avg_contribution_margin_pct']:.1f}% margin")

🔍 Rule 1: Retailer margin below threshold...
  ⚠️  Boots: 14.0% margin
  ⚠️  Asda: -8.5% margin

🔍 Rule 2: AT RISK SKUs...
  ⚠️  SET 72 CAKE CASES VINTAGE CHRISTMAS: -531.2% margin
  ⚠️  WHITE/BLUE PULL BACK RACING CAR: -469.1% margin
  ⚠️  PINK STAR CHRISTMAS DECORATION: -451.2% margin
  ⚠️  BLUE PULL BACK RACING CAR: -412.2% margin
  ⚠️  75 GREEN FAIRY CAKE CASES: -401.9% margin


## Cell 3 — Rule Engine: Trade Spend & Promo Flags

In [0]:
# ── RULE 3: Value-destroying promotions ───────────────────
print("🔍 Rule 3: Value-destroying promotions...")
bad_promos = gold_promo_rank \
    .filter(F.col("roi_flag") == "VALUE DESTROYING") \
    .collect()

for row in bad_promos:
    add_rec(
        rec_type  = "TRADE SPEND",
        area      = f"{row['retailer_name']} — {row['promo_name']}",
        issue     = f"Promotion '{row['promo_name']}' is VALUE DESTROYING with {row['promo_roi_pct']:.1f}% ROI",
        evidence  = f"Trade spend: £{row['trade_spend_gbp']:,.0f} | Incremental margin: £{row['incremental_margin_gbp']:,.0f} | Net loss: £{row['net_incremental_margin_gbp']:,.0f}",
        action    = f"Redesign or cancel '{row['promo_name']}' — reduce discount depth from {row['disc_pct']:.0f}% or shift to Feature/Display mechanics",
        impact    = f"Stopping this promotion saves £{abs(row['net_incremental_margin_gbp']):,.0f} in net margin",
        priority  = "HIGH"
    )
    print(f"  ⚠️  {row['promo_name']}: ROI {row['promo_roi_pct']:.1f}%")

# ── RULE 4: Inefficient trade spend ───────────────────────
print("\n🔍 Rule 4: Inefficient trade spend...")
inefficient = gold_trade_eff \
    .filter(F.col("spend_efficiency_flag") == "INEFFICIENT") \
    .collect()

for row in inefficient:
    spend_per_unit = row["total_spend_per_incremental_unit"]
    avg_roi        = row["avg_roi_pct"]

    # Skip rows where values are None
    if spend_per_unit is None or avg_roi is None:
        continue

    add_rec(
        rec_type  = "TRADE SPEND",
        area      = f"{row['retailer_name']} — {row['promo_type']}",
        issue     = f"Trade spend inefficient for {row['promo_type']} at {row['retailer_name']}",
        evidence  = f"£{spend_per_unit:.2f} per incremental unit (threshold: £5.00) | Avg ROI: {avg_roi:.1f}%",
        action    = f"Reduce {row['promo_type']} investment at {row['retailer_name']} and reallocate to higher-ROI mechanics",
        impact    = "Reallocation of trade budget to efficient channels improves overall ROI",
        priority  = "MEDIUM"
    )
    print(f"  ⚠️  {row['retailer_name']} {row['promo_type']}: £{spend_per_unit:.2f}/unit")

🔍 Rule 3: Value-destroying promotions...
  ⚠️  Boots Xmas Event: ROI -77.9%
  ⚠️  Smyths Summer Sale: ROI -83.7%
  ⚠️  Amazon Lightning Deal: ROI -100.4%
  ⚠️  Boots Baby Event Q3: ROI -118.2%
  ⚠️  Asda Black Friday Baby: ROI -121.7%
  ⚠️  Asda Baby Week: ROI -146.5%
  ⚠️  Asda Back to School: ROI -151.6%
  ⚠️  Boots Baby Event: ROI -163.5%
  ⚠️  Boots Easter Promo: ROI -188.7%
  ⚠️  DTC New Year Sale: ROI -1764.2%

🔍 Rule 4: Inefficient trade spend...


## Cell 4 — Rule Engine: Forecasting & Sell-Through Flags

In [0]:
# ── RULE 5: Poor forecast accuracy ───────────────────────
print("🔍 Rule 5: Forecast accuracy...")
poor_accuracy = gold_accuracy \
    .filter(F.col("accuracy_flag") == "POOR") \
    .collect()

for row in poor_accuracy:
    add_rec(
        rec_type  = "FORECASTING",
        area      = f"Channel: {row['channel']}",
        issue     = f"Forecast accuracy is POOR for {row['channel']} channel",
        evidence  = f"MAPE: {row['mape_pct']:.1f}% | Bias: £{row['bias_gbp']:,.0f} ({row['bias_direction']})",
        action    = f"Review demand drivers for {row['channel']} — incorporate promotional calendar and seasonal factors into forecast model",
        impact    = "Improved forecast accuracy reduces stock risk and improves service level",
        priority  = "MEDIUM"
    )
    print(f"  ⚠️  {row['channel']}: MAPE {row['mape_pct']:.1f}%")

# ── RULE 6: Slow-moving SKUs ─────────────────────────────
print("\n🔍 Rule 6: Slow movers...")
slow_movers = gold_sell_through \
    .filter(F.col("sell_through_flag") == "AT RISK") \
    .groupBy("stock_code", "description", "category") \
    .agg(
        F.avg("sell_through_rate_pct").alias("avg_st_pct"),
        F.max("slow_week_running_total").alias("slow_weeks")
    ) \
    .filter(F.col("slow_weeks") >= 4) \
    .orderBy("avg_st_pct") \
    .limit(5) \
    .collect()

for row in slow_movers:
    add_rec(
        rec_type  = "SELL-THROUGH",
        area      = f"{row['description']} ({row['category']})",
        issue     = f"'{row['description']}' has been AT RISK sell-through for {row['slow_weeks']:.0f}+ weeks",
        evidence  = f"Average sell-through rate: {row['avg_st_pct']:.1f}% (threshold: 40%)",
        action    = f"Run clearance promotion or markdown on '{row['description']}' — prevent stock write-off",
        impact    = "Recovering even 50% of cost on slow movers is better than full write-off",
        priority  = "HIGH" if row["slow_weeks"] >= 8 else "MEDIUM"
    )
    print(f"  ⚠️  {row['description']}: {row['avg_st_pct']:.1f}% ST over {row['slow_weeks']:.0f} weeks")

# ── RULE 7: Seasonal risk ────────────────────────────────
print("\n🔍 Rule 7: Seasonal trough categories...")
trough_cats = gold_seasonal \
    .filter(F.col("seasonal_risk_flag") == "TROUGH — HIGH RISK") \
    .groupBy("category") \
    .agg(F.count("*").alias("trough_periods")) \
    .filter(F.col("trough_periods") >= 2) \
    .collect()

for row in trough_cats:
    add_rec(
        rec_type  = "SEASONAL RISK",
        area      = f"Category: {row['category']}",
        issue     = f"{row['category']} experiences recurring seasonal troughs ({row['trough_periods']} periods below index)",
        evidence  = f"Seasonal index drops below 0.7 in {row['trough_periods']} quarter periods",
        action    = f"Plan targeted promotional investment for {row['category']} during trough periods — consider bundle offers or NPD launch timing",
        impact    = "Smoothing seasonal troughs improves annual revenue predictability",
        priority  = "MEDIUM"
    )
    print(f"  ⚠️  {row['category']}: {row['trough_periods']} trough periods")

🔍 Rule 5: Forecast accuracy...

🔍 Rule 6: Slow movers...
  ⚠️  CLAM SHELL LARGE: 0.2% ST over 11 weeks
  ⚠️  BLUE POLKADOT PUDDING BOWL: 0.3% ST over 14 weeks
  ⚠️  ORIGAMI ROSE INCENSE CONES: 0.8% ST over 4 weeks
  ⚠️  PANTRY CHOPPING BOARD: 0.9% ST over 11 weeks
  ⚠️  ORIGAMI OPIUM INCENSE/CANDLE SET: 1.0% ST over 12 weeks

🔍 Rule 7: Seasonal trough categories...
  ⚠️  Feeding: 2 trough periods
  ⚠️  Other: 2 trough periods
  ⚠️  Travel: 2 trough periods


## Cell 5 — Rule Engine: Pricing & Whitespace Flags

In [0]:
# ── RULE 8: Inelastic categories (price increase opportunity)
print("🔍 Rule 8: Price increase opportunities...")
inelastic = gold_elasticity \
    .filter(F.col("classification") == "INELASTIC") \
    .collect()

for row in inelastic:
    add_rec(
        rec_type  = "PRICING OPPORTUNITY",
        area      = f"Category: {row['category']}",
        issue     = f"{row['category']} is price inelastic — price increase opportunity identified",
        evidence  = f"Elasticity: {row['elasticity_coeff']:.3f} | R²: {row['r_squared']:.3f} | Avg price: £{row['avg_weekly_price']:.2f}",
        action    = f"Test 5-8% RRP increase on {row['category']} in Q1 — monitor volume impact over 4 weeks before full rollout",
        impact    = "5% price increase on inelastic category flows directly to margin with minimal volume loss",
        priority  = "HIGH"
    )
    print(f"  💡 {row['category']}: elasticity {row['elasticity_coeff']:.3f} — PRICE INCREASE OPPORTUNITY")

# ── RULE 9: Major whitespace regions ─────────────────────
print("\n🔍 Rule 9: Whitespace regions...")
major_ws = gold_whitespace \
    .filter(F.col("whitespace_flag") == "MAJOR WHITESPACE") \
    .collect()

for row in major_ws:
    add_rec(
        rec_type  = "WHITESPACE",
        area      = f"Region: {row['uk_region']}",
        issue     = f"{row['uk_region']} is significantly underpenetrated vs population weight",
        evidence  = f"Revenue share: {row['revenue_share_pct']:.1f}% vs population: {row['population_weight_pct']:.1f}% — gap: {row['whitespace_gap_pct']:.1f}%",
        action    = f"Engage new retail partners or increase distribution in {row['uk_region']} — prioritise retailers with strong presence in this region",
        impact    = f"Closing half the whitespace gap in {row['uk_region']} represents meaningful incremental revenue",
        priority  = "MEDIUM"
    )
    print(f"  💡 {row['uk_region']}: {row['whitespace_gap_pct']:.1f}% whitespace gap")

# ── RULE 10: Competitive high risk ───────────────────────
print("\n🔍 Rule 10: Competitive pricing risks...")
comp_risk = gold_competitor \
    .filter(F.col("competitive_risk") == "HIGH RISK") \
    .groupBy("category") \
    .agg(F.avg("price_gap_pct").alias("avg_gap")) \
    .collect()

for row in comp_risk:
    add_rec(
        rec_type  = "COMPETITIVE RISK",
        area      = f"Category: {row['category']}",
        issue     = f"Competitors pricing {abs(row['avg_gap']):.1f}% below us in {row['category']}",
        evidence  = f"Average price gap vs competitors: {row['avg_gap']:.1f}%",
        action    = f"Review {row['category']} price architecture — consider selective price match on top 3 volume SKUs",
        impact    = "Closing the price gap on key SKUs defends market share without blanket price reduction",
        priority  = "HIGH"
    )
    print(f"  ⚠️  {row['category']}: competitors {abs(row['avg_gap']):.1f}% cheaper")

print(f"\n✅ Total recommendations generated: {len(rec_list)}")

🔍 Rule 8: Price increase opportunities...
  💡 Other: elasticity -0.041 — PRICE INCREASE OPPORTUNITY

🔍 Rule 9: Whitespace regions...
  💡 London: 9.7% whitespace gap
  💡 South East: 6.7% whitespace gap

🔍 Rule 10: Competitive pricing risks...
  ⚠️  Healthcare: competitors 24.0% cheaper
  ⚠️  Bathing: competitors 16.6% cheaper

✅ Total recommendations generated: 40


## Cell 6 — Write Gold: Master Recommendations Table

In [0]:
df_recs = spark.createDataFrame(pd.DataFrame(rec_list))
df_recs = df_recs.withColumn("created_at", F.current_timestamp())

(df_recs.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_recommendations"))

print(f"✅ gold_recommendations written: {df_recs.count()} recommendations")
display(df_recs.orderBy("priority", "rec_type"))

✅ gold_recommendations written: 40 recommendations


rec_id,rec_type,area,issue,evidence,recommended_action,commercial_impact,priority,status,created_at
REC-039,COMPETITIVE RISK,Category: Healthcare,Competitors pricing 24.0% below us in Healthcare,Average price gap vs competitors: -24.0%,Review Healthcare price architecture — consider selective price match on top 3 volume SKUs,Closing the price gap on key SKUs defends market share without blanket price reduction,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-040,COMPETITIVE RISK,Category: Bathing,Competitors pricing 16.6% below us in Bathing,Average price gap vs competitors: -16.6%,Review Bathing price architecture — consider selective price match on top 3 volume SKUs,Closing the price gap on key SKUs defends market share without blanket price reduction,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-001,MARGIN,Boots,Contribution margin below 30% threshold at Boots,Average contribution margin: 14.0% (threshold: 30%),Review rebate structure and trade funding terms with Boots — negotiate improved settlement terms or reduce scan allowances,Every 1% margin improvement on this retailer recovers significant P&L value,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-002,MARGIN,Asda,Contribution margin below 30% threshold at Asda,Average contribution margin: -8.5% (threshold: 30%),Review rebate structure and trade funding terms with Asda — negotiate improved settlement terms or reduce scan allowances,Every 1% margin improvement on this retailer recovers significant P&L value,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-036,PRICING OPPORTUNITY,Category: Other,Other is price inelastic — price increase opportunity identified,Elasticity: -0.041 | R²: 0.002 | Avg price: £8.52,Test 5-8% RRP increase on Other in Q1 — monitor volume impact over 4 weeks before full rollout,5% price increase on inelastic category flows directly to margin with minimal volume loss,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-028,SELL-THROUGH,CLAM SHELL LARGE (Healthcare),'CLAM SHELL LARGE' has been AT RISK sell-through for 11+ weeks,Average sell-through rate: 0.2% (threshold: 40%),Run clearance promotion or markdown on 'CLAM SHELL LARGE' — prevent stock write-off,Recovering even 50% of cost on slow movers is better than full write-off,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-029,SELL-THROUGH,BLUE POLKADOT PUDDING BOWL (Toys),'BLUE POLKADOT PUDDING BOWL' has been AT RISK sell-through for 14+ weeks,Average sell-through rate: 0.3% (threshold: 40%),Run clearance promotion or markdown on 'BLUE POLKADOT PUDDING BOWL' — prevent stock write-off,Recovering even 50% of cost on slow movers is better than full write-off,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-031,SELL-THROUGH,PANTRY CHOPPING BOARD (Healthcare),'PANTRY CHOPPING BOARD' has been AT RISK sell-through for 11+ weeks,Average sell-through rate: 0.9% (threshold: 40%),Run clearance promotion or markdown on 'PANTRY CHOPPING BOARD' — prevent stock write-off,Recovering even 50% of cost on slow movers is better than full write-off,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-032,SELL-THROUGH,ORIGAMI OPIUM INCENSE/CANDLE SET (Other),'ORIGAMI OPIUM INCENSE/CANDLE SET' has been AT RISK sell-through for 12+ weeks,Average sell-through rate: 1.0% (threshold: 40%),Run clearance promotion or markdown on 'ORIGAMI OPIUM INCENSE/CANDLE SET' — prevent stock write-off,Recovering even 50% of cost on slow movers is better than full write-off,HIGH,OPEN,2026-03-18T13:11:52.796Z
REC-003,SKU PORTFOLIO,SET 72 CAKE CASES VINTAGE CHRISTMAS (Nursery),SKU 'SET 72 CAKE CASES VINTAGE CHRISTMAS' is AT RISK with -531.2% margin,Revenue rank: 3806 | Units sold: 672 | Margin rank: 5571,Review cost structure or delist 'SET 72 CAKE CASES VINTAGE CHRISTMAS' — consider range rationalisation,Removing loss-making SKUs improves portfolio average margin,HIGH,OPEN,2026-03-18T13:11:52.796Z


## Cell 7 — Generate Weekly Trading Briefing
Auto-generates a structured markdown briefing from Gold KPIs
Mimics a real UK commercial trading review pack

In [0]:
# Pull headline KPIs
kpis = spark.table(f"{DATABASE}.silver_pos_cleaned").agg(
    F.round(F.sum("gross_revenue_gbp"), 0).alias("gross_revenue"),
    F.round(F.sum("net_revenue_gbp"), 0).alias("net_revenue"),
    F.round(F.sum("contribution_margin_gbp"), 0).alias("contribution_margin"),
    F.round(F.avg("contribution_margin_pct"), 1).alias("margin_pct"),
    F.round(F.sum("total_margin_leakage_gbp"), 0).alias("leakage")
).collect()[0]

# Top retailer
top_retailer = spark.table(f"{DATABASE}.gold_retailer_pl") \
    .groupBy("retailer_name") \
    .agg(F.sum("gross_revenue_gbp").alias("rev")) \
    .orderBy(F.col("rev").desc()) \
    .first()

# Best promo
best_promo = spark.table(f"{DATABASE}.gold_promo_roi_ranking") \
    .orderBy("roi_rank").first()

# Worst promo
worst_promo = spark.table(f"{DATABASE}.gold_promo_roi_ranking") \
    .orderBy(F.col("promo_roi_pct").asc()).first()

# High priority recs count
high_recs = spark.table(f"{DATABASE}.gold_recommendations") \
    .filter(F.col("priority") == "HIGH").count()

# Build briefing text
briefing_date = datetime.now().strftime("%d %B %Y")

briefing = f"""
# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
## Weekly Trading Briefing — {briefing_date}
### Prepared by: Byron Kamau | Data Analyst

---

## 1. COMMERCIAL HEADLINE PERFORMANCE

| KPI | Value |
|---|---|
| Total Gross Revenue | £{kpis['gross_revenue']:,.0f} |
| Total Net Revenue | £{kpis['net_revenue']:,.0f} |
| Contribution Margin | £{kpis['contribution_margin']:,.0f} |
| Avg Contribution Margin % | {kpis['margin_pct']:.1f}% |
| Total Margin Leakage | £{kpis['leakage']:,.0f} |

**Commercial Context:**
Net revenue is £{kpis['gross_revenue'] - kpis['net_revenue']:,.0f} below gross revenue,
reflecting trade funding deductions including rebates, settlement discounts,
and scan allowances. Contribution margin of {kpis['margin_pct']:.1f}% is
{'above' if kpis['margin_pct'] >= 40 else 'below'} the 40% target threshold.

---

## 2. RETAILER PERFORMANCE

**Top Performing Retailer:** {top_retailer['retailer_name']} with £{top_retailer['rev']:,.0f} gross revenue.

See gold_retailer_pl for full retailer P&L breakdown including
margin by retailer and margin leakage waterfall.

---

## 3. PROMOTIONAL PERFORMANCE

**Best ROI Promotion:** {best_promo['promo_name']} at {best_promo['retailer_name']}
- ROI: {best_promo['promo_roi_pct']:.1f}% | Flag: {best_promo['roi_flag']}

**Worst ROI Promotion:** {worst_promo['promo_name']} at {worst_promo['retailer_name']}
- ROI: {worst_promo['promo_roi_pct']:.1f}% | Flag: {worst_promo['roi_flag']}

**Action Required:** Review value-destroying promotions before next trading period.

---

## 4. FORECAST & RISK

Forecast models trained per channel using Prophet time-series.
See gold_forecast_scenarios for Base / Upside / Downside scenario outputs.
See gold_sell_through for SKU-level sell-through tracking and slow-mover flags.
See gold_seasonal_risk for category seasonal risk index by quarter.

---

## 5. PRICING & MARKET INTELLIGENCE

Price elasticity calculated for all 8 product categories using OLS regression.
Inelastic categories present price increase opportunities.
Competitor pricing index built for 5 key UK nursery competitors.
Whitespace analysis identifies underpenetrated UK regions.

---

## 6. PRIORITY ACTIONS THIS WEEK

**{high_recs} HIGH priority recommendations require immediate attention.**
See gold_recommendations table for full list with evidence and actions.

Top 3 actions:
1. Review and redesign value-destroying promotions
2. Address AT RISK SKUs — markdown or delist decisions required
3. Investigate price increase opportunity on inelastic categories

---

*This briefing is auto-generated from the UK Retail Commercial Intelligence Platform.*
*All figures sourced from Delta Gold tables as at {briefing_date}.*
"""

print(briefing)

# Save briefing as a Delta table
df_briefing = spark.createDataFrame(
    pd.DataFrame([{
        "briefing_date": briefing_date,
        "briefing_text": briefing,
        "high_priority_recs": high_recs,
        "gross_revenue_gbp": float(kpis["gross_revenue"]),
        "contribution_margin_pct": float(kpis["margin_pct"]),
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }])
)

(df_briefing.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{DATABASE}.gold_weekly_briefing"))

print("\n✅ gold_weekly_briefing written")


# UK RETAIL COMMERCIAL INTELLIGENCE PLATFORM
## Weekly Trading Briefing — 18 March 2026
### Prepared by: Byron Kamau | Data Analyst

---

## 1. COMMERCIAL HEADLINE PERFORMANCE

| KPI | Value |
|---|---|
| Total Gross Revenue | £17,870,978 |
| Total Net Revenue | £15,019,197 |
| Contribution Margin | £8,600,494 |
| Avg Contribution Margin % | 45.0% |
| Total Margin Leakage | £2,851,781 |

**Commercial Context:**
Net revenue is £2,851,781 below gross revenue,
reflecting trade funding deductions including rebates, settlement discounts,
and scan allowances. Contribution margin of 45.0% is
above the 40% target threshold.

---

## 2. RETAILER PERFORMANCE

**Top Performing Retailer:** DTC Ecommerce with £5,705,082 gross revenue.

See gold_retailer_pl for full retailer P&L breakdown including
margin by retailer and margin leakage waterfall.

---

## 3. PROMOTIONAL PERFORMANCE

**Best ROI Promotion:** JL Christmas Nursery at John Lewis
- ROI: 871.6% | Flag: STRONG POSITIVE

**Worst ROI Promoti

## Cell 8 — Final Quality Check: All Gold Tables

In [0]:
print("=" * 65)
print("  MODULE 6 — FINAL PLATFORM QUALITY CHECK")
print("=" * 65)

all_gold_tables = [
    # Module 1
    "silver_pos_cleaned",
    "gold_retailer_pl",
    "gold_sku_profitability",
    "gold_regional_performance",
    "gold_margin_waterfall",
    # Module 2
    "gold_promo_events",
    "gold_trade_spend_efficiency",
    "gold_promo_roi_ranking",
    # Module 3
    "gold_forecast_scenarios",
    "gold_forecast_accuracy",
    "gold_sell_through",
    "gold_seasonal_risk",
    # Module 4
    "gold_price_elasticity",
    "gold_competitor_pricing",
    "gold_whitespace",
    "gold_pricing_recommendations",
    # Module 6
    "gold_recommendations",
    "gold_weekly_briefing",
]

total_tables = 0
total_rows   = 0

for t in all_gold_tables:
    try:
        n = spark.sql(
            f"SELECT COUNT(*) AS n FROM {DATABASE}.{t}"
        ).collect()[0]["n"]
        print(f"  ✅  {t:<40} {n:>10,} rows")
        total_tables += 1
        total_rows   += n
    except Exception as e:
        print(f"  ❌  {t:<40} ERROR")

print("=" * 65)
print(f"  📊 Total tables  : {total_tables}")
print(f"  📊 Total rows    : {total_rows:,}")
print("=" * 65)

# Recommendations summary
print()
print("  🎯 RECOMMENDATIONS SUMMARY")
print("  " + "-" * 55)
spark.sql(f"""
    SELECT priority, rec_type, COUNT(*) AS count
    FROM {DATABASE}.gold_recommendations
    GROUP BY priority, rec_type
    ORDER BY
        CASE priority
            WHEN 'HIGH'   THEN 1
            WHEN 'MEDIUM' THEN 2
            ELSE 3
        END,
        rec_type
""").show(truncate=False)

print("=" * 65)
print()
print("  🏆 PLATFORM BUILD COMPLETE!")
print()
print("  ✅ Module 7  — Bronze Ingestion & ERP Foundation")
print("  ✅ Module 1  — Silver Cleaning & Commercial P&L")
print("  ✅ Module 2  — Trade Spend & Promotional ROI")
print("  ✅ Module 3  — Forecasting & Sell-Through")
print("  ✅ Module 4  — Pricing Elasticity & Market Intelligence")
print("  ✅ Module 6  — Recommendations Engine")
print()
print("  👉 Next: Module 5 — Power BI Dashboard")
print("=" * 65)

  MODULE 6 — FINAL PLATFORM QUALITY CHECK
  ✅  silver_pos_cleaned                          958,501 rows
  ✅  gold_retailer_pl                                200 rows
  ✅  gold_sku_profitability                        5,571 rows
  ✅  gold_regional_performance                       275 rows
  ✅  gold_margin_waterfall                             7 rows
  ✅  gold_promo_events                                12 rows
  ✅  gold_trade_spend_efficiency                       7 rows
  ✅  gold_promo_roi_ranking                           12 rows
  ✅  gold_forecast_scenarios                         696 rows
  ✅  gold_forecast_accuracy                            2 rows
  ✅  gold_sell_through                           208,957 rows
  ✅  gold_seasonal_risk                               81 rows
  ✅  gold_price_elasticity                             9 rows
  ✅  gold_competitor_pricing                          45 rows
  ✅  gold_whitespace                                  11 rows
  ✅  gold_pricing_recommenda